In [117]:
from dotenv import load_dotenv
import os, sys
load_dotenv()
print("OPENAI_API_KEY present:", bool(os.environ.get("OPENAI_API_KEY")))
print("OPENAI_API_BASE:", os.environ.get("OPENAI_API_BASE"))

OPENAI_API_KEY present: True
OPENAI_API_BASE: https://chat-ai.academiccloud.de/v1


## Install libraries

In [118]:
! pip install -q langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv
! pip install -U langchain-classic langchain-community 

! pip install ragas
! pip install pandas
! pip install flashrank


[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [119]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [120]:
# Loading the folder with mutliple pdfs at once.

all_documents = []
pdf_folder = "Bacteremia_Guidelines/"
for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, filename))
        all_documents.extend(loader.load())

In [121]:
print(all_documents[50].metadata)  # Display the first 500 characters of the first document

{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-05-15T13:21:59-04:00', 'author': 'Dejene, Sara Zufan', 'moddate': '2026-05-15T13:21:59-04:00', 'source': 'Bacteremia_Guidelines/UNC_GN‑bacteremia pathway..pdf', 'total_pages': 6, 'page': 5, 'page_label': '6'}


# Preprocessing the Docs -- References 

In [122]:
def is_reference_page(page_text):
    markers = ["REFERENCES", "doi:10.", "Clin Infect Dis.", "et al."]
    hits = sum(1 for m in markers if m in all_documents)
    return hits >= 2

filtered_pages = [p for p in all_documents if not is_reference_page(p.page_content)]

## Step 1b - Indexing (Text Splitting)

In [123]:
from langchain_openai import OpenAIEmbeddings

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=800,
#     chunk_overlap=150,
#    separators=["\n\n", "\n", ". ", " ", ""] )

embeddings = OpenAIEmbeddings(    
    model="e5-mistral-7b-instruct",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    openai_api_base=os.environ["OPENAI_API_BASE"],
    tiktoken_enabled=False,
    check_embedding_ctx_length=False
    
)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85
)

chunks = semantic_splitter.split_documents(filtered_pages) # all filtered docs pages page content is split into chunks of 500 characters with an overlap of 200 characters

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.acad

In [124]:
len(chunks)

193

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [125]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv(override=True)

vector_store = FAISS.from_documents(chunks, embeddings)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"


In [126]:
vector_store.index_to_docstore_id

{0: '5d555cf6-a229-4d4e-af12-4efe64214af4',
 1: 'd4009262-6689-4745-9427-319bd61385c3',
 2: '8577b312-0434-4160-97e4-27fc42b86fd9',
 3: '739862ff-1ef6-4ea2-a719-60731c343dc8',
 4: '6def5000-42d7-4edd-87af-ec531844e1a1',
 5: '2095c4b7-c93f-40ee-b963-3a68c2b22cdc',
 6: 'a10172e2-27a0-4ab1-b18a-88e9d308d263',
 7: 'e0abf7e1-c531-418f-bd06-6326f9d782bc',
 8: '79262ec6-9868-4cbb-bb51-ca45fb24db14',
 9: 'd6fcd606-fed3-4eb8-a823-fe74726b20fd',
 10: 'f1318c46-1997-4f62-b721-0768039c9c68',
 11: '0807ae9b-56e7-45d8-9c1f-8d113ff2e139',
 12: '5af2b945-ff10-42cd-a1b3-20a5c2089f4d',
 13: '7324ce9a-6be9-496f-8dce-a58da46d972b',
 14: 'eeaf15f8-e625-4a7b-8a7c-b87222d5f1ed',
 15: 'a5061b90-55a9-409a-b118-6f8670c453ce',
 16: '4dfeb818-a0f5-4dd6-81c8-85e88d2bfc77',
 17: '810d103b-f741-4fcc-97d4-a98d5b51f4f2',
 18: 'a0c1cdf2-db60-4b1c-849f-de9e9c2d0c88',
 19: 'fb35af70-95ac-4ce7-9199-b289a854a34c',
 20: '12892c5b-89a8-447e-a30c-33d6dbb2a2e6',
 21: '5f3c4317-23c2-4df3-a5bb-02ff63f579d8',
 22: 'cfc0141f-e31a-

In [127]:
vector_store.get_by_ids(['6efe3678-80ae-4d30-8784-9c8d3c21b50a'])

[]

## Step 2 - Retrieval  [input is Query and output is List of Docs]

In [128]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 10}) 


In [129]:
gwdg_llm = ChatOpenAI(
    model="qwen3-30b-a3b-instruct-2507",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    openai_api_base=os.environ["OPENAI_API_BASE"],
    temperature=0
)

In [130]:
# MQR bcz the retreiver cant able to retrive teh content related to teh question bcz question contains the synonyms of the content actually in docs,
# this MRQ retriever will generate multiple queries for the same question and then use the retriever to get the relevant content from the docs.

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate

query_gen_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Generate exactly 3 alternative phrasings of this clinical question, using standard medical terminology only. Keep each rewrite concise and literal, avoid creative rephrasing.

Question: {question}

Alternative phrasings:"""
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=gwdg_llm,
    prompt=query_gen_prompt
)

In [131]:
docs = multi_query_retriever.invoke("Does immunosuppression alone make a bacteremia case complicated?")
for i, doc in enumerate(docs):
    print(f"--- Doc {i} ---")
    print(doc.page_content)
    print()

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Is bacteremia considered complicated when immunosuppression is the sole underlying condition?  ', '2. Does immunosuppression alone constitute a risk factor for complicated bacteremia?  ', '3. In the absence of other comorbidities, does immunosuppression alone lead to complicated bacteremia?']
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"


--- Doc 0 ---
DEFINITION 
We define uncomplicated bacteremia as follows:1–7 
1. Monomicrobial (excluding Brucella or Salmonella) 
2. Bloodstream infection secondary to urinary tract infection, indwelling intravascular catheter, intra-
abdominal or biliary infection, pneumonia (without structural lung disease, empyema, abscess, or cystic 
fibrosis), or skin and soft tissue infection. 3. No evidence of endocarditis, endovascular disease, central nervous system infection or bone/joint 
infection. 4. Source control achieved (e.g., abscess drained, infected catheter(s) removed, biliary obstruction 
resolved) 
5.

--- Doc 1 ---
Always identify the 
likely source of bacteremia and proceed with 
source control whenever possible if a lingering 
source has been identified.

--- Doc 2 ---
All 
removable devices should be removed and all 
abscesses should be debrided. Infectious diseases 
consultation has been demonstrated to improve 
outcomes in patients with Staph aureus 
bacteremia and should b

In [132]:
# Specialized model to filterout which chunks (in total 10) are highly releveant to question

# Fixed: Moved from langchain.retrievers to langchain_classic.retrievers
from langchain_classic.retrievers import ContextualCompressionRetriever

# Third-party integrations like Flashrank are imported from langchain_community
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank(top_n=3)

final_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=multi_query_retriever
)


In [133]:
fin_docs = final_retriever.invoke("Does immunosuppression alone make a bacteremia case complicated?")
for doc in fin_docs:
    print(doc.page_content)
    print("---")

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Is bacteremia considered complicated when immunosuppression is the sole underlying condition?  ', '2. Does immunosuppression alone constitute a risk factor for complicated bacteremia?  ', '3. In the absence of other comorbidities, does immunosuppression alone lead to complicated bacteremia?']
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"


One study also reported shorter time to return to baseline 
with a shorter antibiotic course2. One retrospective study revealed increased odds of recurrence in 
patients with bacteremia associated with complicated UTI in patients treated with shorter courses of 
antibiotics (7 vs. 14 days), however, this difference resolved when only patients on highly bioavailable 
oral agents were included22. Bacteremia associated with pyelonephritis may require a longer duration of 
10-14 days if utilizing trimethoprim-sulfamethoxazole or oral beta-lactam antibiotics12. Most studies exclude patients with immunocompromising conditions, but some data are available. Immunocompromised patients have been represented in small numbers in RCTs, with no difference in 
mortality, bloodstream infection relapse, or rehospitalization noted in systematic review compared to 
immunocompetent patients1,2,23. One prospective observational study evaluated long versus short 
courses of antibiotics for GN-BSI in immunoc

## Step 3 - Augmentation

In [134]:
llm = ChatOpenAI(model="qwen3-30b-a3b-instruct-2507", temperature=0)


In [135]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a clinical guideline assistant for infectious disease and bacteremia questions. This tool is for research and prototyping only, not for clinical decision-making.

Rules you must follow:
1. Answer only using the provided reference documents below. Do not use outside knowledge.
2. Do not combine or synthesize information across different retrieved chunks into a single claim, unless that exact combined claim is explicitly stated together in one source. If multiple chunks are relevant but address different aspects or scenarios, present them as separate points with their own citations rather than merging them into one unified statement.
3. If two sources appear related but address different clinical situations, state this distinction clearly rather than merging them.
4. If the answer is not explicitly stated in the references, say so honestly instead of inferring or guessing.
5. If the retrieved context does not contain information directly relevant to the question, explicitly state that the provided guidelines don't address this, rather than attempting to answer from general knowledge.
6. For every claim, cite the source using the exact filename and page number shown in the context metadata, formatted exactly as [filename, p.X]. Replace 'filename' and 'X' with the real values found in the context. Never output the literal placeholder word SOURCE.

Reference documents:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

In [136]:
question          = "This patients bacteremia source isnt obvious — how do I decide if this counts as uncomplicated before choosing a duration?"
retrieved_docs    = final_retriever.invoke(question)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. In a patient with bacteremia of unclear origin, how is uncomplicated bacteremia defined to guide duration of therapy?  ', '2. For bacteremia with an obscure source, what criteria determine whether it is classified as uncomplicated to inform treatment duration?  ', '3. How should one classify bacteremia as uncomplicated when the source is not apparent to guide appropriate treatment duration?']
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"


In [137]:
retrieved_docs

[Document(metadata={'id': 10, 'relevance_score': np.float32(0.999236), 'producer': 'Adobe PDF Library 26.1.119', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-10T16:05:45-05:00', 'author': 'UNMC;mackenzie.keintz@unmc.edu', 'category': '', 'comments': '', 'company': '', 'keywords': '', 'manager': '', 'moddate': '2026-04-10T16:07:15-05:00', 'sourcemodified': '', 'subject': '', 'title': 'Guidance on management of uncomplicated bloodstream infections from Gram-negative organisms', 'source': 'Bacteremia_Guidelines/gram_negative_bacteremia_guidance_8-2023.pdf', 'total_pages': 7, 'page': 5, 'page_label': '6'}, page_content='UNMC / NEBRASKA MEDICINE  |  6 \n \nWhen clinical uncertainty exists regarding adequate source control of a GN-BSI in immunocompromised \npersons, infectious diseases consultation is recommended. Repeat Blood Cultures: \nExperts agree and the available data support that repeat blood cultures are unnecessary in patients with \nGNB if they are clinicall

In [138]:
# Instead of sending the 4 docs seperately join all docs page content into a single string with a separator of 2 new lines. This will be used as the context for the prompt template.

context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'UNMC / NEBRASKA MEDICINE  |  6 \n \nWhen clinical uncertainty exists regarding adequate source control of a GN-BSI in immunocompromised \npersons, infectious diseases consultation is recommended. Repeat Blood Cultures: \nExperts agree and the available data support that repeat blood cultures are unnecessary in patients with \nGNB if they are clinically improving on therapy10. In one study, persistent bacteremia was associated with \nendovascular infection and inability to obtain source control at 48 hours. Follow up blood cultures were \nlow yield in patients with GN-BSI25. In addition, several retrospective evaluations of follow-up blood \ncultures have demonstrated prolonged length of stay and antibiotic duration26,27. Situations where repeat \ncultures may be appropriate include patients without an appropriate clinical response in 72 hours on \nappropriate antibiotic therapy, in patients with concern for endocarditis or endovascular infections, or \npatients in which adequate sourc

In [139]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [140]:
final_prompt

StringPromptValue(text="You are a clinical guideline assistant for infectious disease and bacteremia questions. This tool is for research and prototyping only, not for clinical decision-making.\n\nRules you must follow:\n1. Answer only using the provided reference documents below. Do not use outside knowledge.\n2. Do not combine or synthesize information across different retrieved chunks into a single claim, unless that exact combined claim is explicitly stated together in one source. If multiple chunks are relevant but address different aspects or scenarios, present them as separate points with their own citations rather than merging them into one unified statement.\n3. If two sources appear related but address different clinical situations, state this distinction clearly rather than merging them.\n4. If the answer is not explicitly stated in the references, say so honestly instead of inferring or guessing.\n5. If the retrieved context does not contain information directly relevant to

## Step 4 - Generation

## Building a Chain

In [141]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [142]:
def format_docs_with_citations(docs): # citates the source and page number for each document in the retrieved docs and returns a single string with all the docs page content joined together with 2 new lines as separator.
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        formatted.append(f"[Source: {source}, p.{page}]\n{doc.page_content}")
    return "\n\n".join(formatted)

In [143]:
formatted_context = format_docs_with_citations(retrieved_docs)
formatted_context

'[Source: Bacteremia_Guidelines/gram_negative_bacteremia_guidance_8-2023.pdf, p.5]\nUNMC / NEBRASKA MEDICINE  |  6 \n \nWhen clinical uncertainty exists regarding adequate source control of a GN-BSI in immunocompromised \npersons, infectious diseases consultation is recommended. Repeat Blood Cultures: \nExperts agree and the available data support that repeat blood cultures are unnecessary in patients with \nGNB if they are clinically improving on therapy10. In one study, persistent bacteremia was associated with \nendovascular infection and inability to obtain source control at 48 hours. Follow up blood cultures were \nlow yield in patients with GN-BSI25. In addition, several retrospective evaluations of follow-up blood \ncultures have demonstrated prolonged length of stay and antibiotic duration26,27. Situations where repeat \ncultures may be appropriate include patients without an appropriate clinical response in 72 hours on \nappropriate antibiotic therapy, in patients with concern

In [144]:
parallel_chain = RunnableParallel({
    'context': final_retriever | RunnableLambda(format_docs_with_citations), # all functions should be lambda so used runnable lambda
    'question': RunnablePassthrough() # no need to format the question so used runnable passthrough
})

In [145]:
# parallel_chain.invoke('This patients bacteremia source isnt obvious — how do I decide if this counts as uncomplicated before choosing a duration?')

# got the context, but its based on semantic meaning of the question and not the exact question. So to get the exact question we need to use the prompt template to format the context and question into a single prompt.

In [146]:
parser = StrOutputParser()

In [147]:
main_chain = parallel_chain | prompt | llm | parser

In [148]:
# answer = main_chain.invoke('Explain about the coronavirus disease and advantages of it?')


# Evaluation - RAGAS

In [149]:
import time

def invoke_with_retry(chain, question, max_retries=5, wait=3):
    for attempt in range(max_retries):
        try:
            return chain.invoke(question)
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(wait * (attempt + 1))
    raise RuntimeError(f"Failed after {max_retries} attempts: {question}")

In [150]:
# test set - to get answers and contexts

import pandas as pd

df = pd.read_csv("bacteremia_eval_testset.csv")

answers = []
contexts = []

for question in df["question"]:
    result = invoke_with_retry(main_chain, question)
    answers.append(result) # result is just a string of answer, not contants any metadata.
    retrieved_docs = invoke_with_retry(final_retriever, question)
    contexts.append([doc.page_content for doc in retrieved_docs])
    time.sleep(2)


df["answer"] = answers
df["contexts"] = contexts

df.to_csv("bacteremia_eval_with_answers_v9.csv", index=False)




INFO:openai._base_client:Retrying request to /chat/completions in 0.433188 seconds


INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. In a patient with culture-proven bacteremia and no identifiable source, how is uncomplicated bacteremia defined to guide antibiotic duration?  ', '2. How should uncomplicated bacteremia be determined in a patient with occult source bacteremia to inform treatment duration?  ', '3. For a patient with bacteremia and no apparent source, what criteria define uncomplicated disease to guide antimicrobial therapy duration?']
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Reques

In [151]:
import sys
import types

# 1. Forge a blank module structure in memory
fake_vertex_module = types.ModuleType("langchain_community.chat_models.vertexai")

# 2. Add an empty class inside it to satisfy Ragas' internal search
class DummyChatVertexAI:
    pass

fake_vertex_module.ChatVertexAI = DummyChatVertexAI

# 3. Mount it to Python's system registry so it is instantly available
sys.modules["langchain_community.chat_models.vertexai"] = fake_vertex_module

In [152]:
from ragas import EvaluationDataset
from datasets import Dataset

eval_data = {
    "question": df["question"].tolist(),
    "answer": df["answer"].tolist(),
    "contexts": df["contexts"].tolist(),
    "ground_truth": df["ground_truth"].tolist(),
}

eval_dataset = Dataset.from_dict(eval_data)

In [153]:
# Gwdg llm not open ai 

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import os

gwdg_llm = ChatOpenAI(
    model="qwen3-30b-a3b-instruct-2507",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    openai_api_base=os.environ["OPENAI_API_BASE"],
    temperature=0
)

gwdg_embeddings = OpenAIEmbeddings(
    model="e5-mistral-7b-instruct",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    openai_api_base=os.environ["OPENAI_API_BASE"],
    tiktoken_enabled=False,
    check_embedding_ctx_length=False
)

ragas_llm = LangchainLLMWrapper(gwdg_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(gwdg_embeddings)

C:\Users\sunil\AppData\Local\Temp\ipykernel_9352\2967234579.py:23: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(gwdg_llm)
C:\Users\sunil\AppData\Local\Temp\ipykernel_9352\2967234579.py:24: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(gwdg_embeddings)


In [154]:
# import time
# from ragas import evaluate, RunConfig
# from ragas.metrics import faithfulness, context_precision, context_recall, answer_relevancy



# def run_evaluation_with_cooldown(dataset, metrics, llm, embeddings, max_attempts=5):
#     for attempt in range(max_attempts):
#         try:
#             return evaluate(
#                 dataset,
#                 metrics=metrics,
#                 llm=llm,
#                 embeddings=embeddings,
#                 run_config=RunConfig(max_workers=1, timeout=180, max_retries=3) #  LLM judge requests GWDG 1 onte time instead of parallel hits
# )
#         except Exception as e:
#             wait_time = 60 * (attempt + 1)
#             print(f"Attempt {attempt+1} failed: {e}. Waiting {wait_time}s before retry.")
#             time.sleep(wait_time)
#     raise RuntimeError("All attempts failed.")


# results = run_evaluation_with_cooldown(eval_dataset, [faithfulness, context_precision, context_recall, answer_relevancy], ragas_llm, ragas_embeddings)


# print(results)

In [161]:
import os
import time
import json
import pandas as pd
from datetime import datetime
from ragas import evaluate, RunConfig
from ragas.metrics import faithfulness, context_precision, context_recall, answer_relevancy


# ------------------------------------------------------------------
# STEP 1: Run ONE evaluation attempt, retrying if it fails
# ------------------------------------------------------------------
def run_evaluation_with_cooldown(dataset, metrics, llm, embeddings, max_attempts=5, base_wait=60):
    for attempt in range(max_attempts):
        try:
            result = evaluate(
                dataset,
                metrics=metrics,
                llm=llm,
                embeddings=embeddings,
                run_config=RunConfig(
                    max_workers=1,   # send requests one at a time, no parallel bursts to GWDG
                    timeout=180,     # allow up to 180s per request before giving up
                    max_retries=5    # let ragas itself retry internally a few times too
                )
            )
            return result
        except Exception as e:
            wait_time = base_wait * (attempt + 1)   # 60s, 120s, 180s, 240s, 300s
            print(f"  Attempt {attempt+1}/{max_attempts} failed: {e}")
            print(f"  Waiting {wait_time}s before retry...")
            time.sleep(wait_time)
    raise RuntimeError("All attempts failed for this evaluation run.")

C:\Users\sunil\AppData\Local\Temp\ipykernel_9352\1496150620.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, context_recall, answer_relevancy
C:\Users\sunil\AppData\Local\Temp\ipykernel_9352\1496150620.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision, context_recall, answer_relevancy
C:\Users\sunil\AppData\Local\Temp\ipykernel_9352\1496150620.py:7: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from rag

In [162]:
# ------------------------------------------------------------------
# STEP 2: Run the FULL evaluation multiple times (n_runs), with
#         cooldown BETWEEN runs + checkpoint saving after each run
# ------------------------------------------------------------------
def run_multi_trial_evaluation(
    dataset,
    metrics,
    llm,
    embeddings,
    n_runs=5,
    cooldown_between_runs=90,   # rest period between full runs, not just retries
    checkpoint_file="ragas_multi_run_checkpoint.json"
):
    all_results = []

    # Resume automatically if a previous partial run exists
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            saved = json.load(f)
        all_results = saved.get("runs", [])
        print(f"Resuming: {len(all_results)} run(s) already completed and saved.")

    remaining_runs = n_runs - len(all_results)

    for i in range(remaining_runs):
        run_number = len(all_results) + 1
        print(f"\n=== Starting evaluation run {run_number}/{n_runs} at {datetime.now().strftime('%H:%M:%S')} ===")

        result = run_evaluation_with_cooldown(dataset, metrics, llm, embeddings)
        result_df = result.to_pandas()

        run_scores = {
            "run": run_number,
            "timestamp": datetime.now().isoformat(),
            "faithfulness": float(result_df["faithfulness"].mean()),
            "context_precision": float(result_df["context_precision"].mean()),
            "context_recall": float(result_df["context_recall"].mean()),
            "answer_relevancy": float(result_df["answer_relevancy"].mean()),
        }
        all_results.append(run_scores)

        # Save progress immediately -- so a later crash doesn't lose earlier runs
        with open(checkpoint_file, "w") as f:
            json.dump({"runs": all_results}, f, indent=2)

        print(f"Run {run_number} scores: {run_scores}")

        if run_number < n_runs:
            print(f"Cooling down {cooldown_between_runs}s before next run...")
            time.sleep(cooldown_between_runs)

    return pd.DataFrame(all_results)


In [163]:


# ------------------------------------------------------------------
# STEP 3: ACTUALLY RUN IT  (this replaces your old "results = ..." line)
# ------------------------------------------------------------------
summary_df = run_multi_trial_evaluation(
    dataset=eval_dataset,                # <-- your existing dataset variable
    metrics=[faithfulness, context_precision, context_recall, answer_relevancy],
    llm=ragas_llm,                        # <-- your existing ragas_llm variable
    embeddings=ragas_embeddings,          # <-- your existing ragas_embeddings variable
    n_runs=5,
    cooldown_between_runs=90
)

print(summary_df)

# Save per-run results
summary_df.to_csv("ragas_multi_run_results.csv", index=False)

# Compute and save final mean +/- std across all 5 runs
final_stats = summary_df[["faithfulness", "context_precision", "context_recall", "answer_relevancy"]].agg(["mean", "std"])
print(final_stats)
final_stats.to_csv("ragas_final_mean_std.csv")


=== Starting evaluation run 1/5 at 13:30:18 ===


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   2%|▎         | 1/40 [00:09<06:26,  9.92s/it]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   8%|▊         | 3/40 [00:14<02:28,  4.01s/it]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
Evaluating:  10%|█         | 4/40 [00:

Run 1 scores: {'run': 1, 'timestamp': '2026-07-11T13:32:37.164358', 'faithfulness': 0.7169841269841271, 'context_precision': 0.8333333332833334, 'context_recall': 0.9666666666666666, 'answer_relevancy': 0.7533888484520431}
Cooling down 90s before next run...

=== Starting evaluation run 2/5 at 13:34:07 ===


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   2%|▎         | 1/40 [00:09<06:10,  9.51s/it]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   8%|▊         | 3/40 [00:15<02:35,  4.20s/it]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/embeddings "HTTP/1.1 200 OK"
Evaluating:  10%|█         | 4/40 [00:

Run 2 scores: {'run': 2, 'timestamp': '2026-07-11T13:36:29.575338', 'faithfulness': 0.7526984126984126, 'context_precision': 0.89999999995, 'context_recall': 0.9666666666666666, 'answer_relevancy': 0.7510937845292884}
Cooling down 90s before next run...

=== Starting evaluation run 3/5 at 13:37:59 ===


...

Run 3 scores: {'run': 3, 'timestamp': '2026-07-11T13:46:13.738913', 'faithfulness': 0.7978835978835979, 'context_precision': 0.9999999999333333, 'context_recall': 0.8333333333333333, 'answer_relevancy': 0.9147218308838866}
Cooling down 90s before next run...

=== Starting evaluation run 4/5 at 13:47:43 ===


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.473414 seconds
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.943777 seconds
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.444716 seconds
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.958265 seconds
INFO:httpx:HTTP Request: POST https://

Run 4 scores: {'run': 4, 'timestamp': '2026-07-11T13:58:16.526812', 'faithfulness': nan, 'context_precision': nan, 'context_recall': nan, 'answer_relevancy': nan}
Cooling down 90s before next run...

=== Starting evaluation run 5/5 at 13:59:46 ===


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 12.000000 seconds
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   2%|▎         | 1/40 [00:27<17:49, 27.42s/it]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating:   8%|▊         | 3/40 [00:33<05:06,  8.28s/it]INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POS

Run 5 scores: {'run': 5, 'timestamp': '2026-07-11T14:02:29.803686', 'faithfulness': 0.7201984126984127, 'context_precision': 0.8499999999508333, 'context_recall': 0.9666666666666666, 'answer_relevancy': 0.7534995983401596}
   run                   timestamp  faithfulness  context_precision  \
0    1  2026-07-11T13:32:37.164358      0.716984           0.833333   
1    2  2026-07-11T13:36:29.575338      0.752698           0.900000   
2    3  2026-07-11T13:46:13.738913      0.797884           1.000000   
3    4  2026-07-11T13:58:16.526812           NaN                NaN   
4    5  2026-07-11T14:02:29.803686      0.720198           0.850000   

   context_recall  answer_relevancy  
0        0.966667          0.753389  
1        0.966667          0.751094  
2        0.833333          0.914722  
3             NaN               NaN  
4        0.966667          0.753500  
      faithfulness  context_precision  context_recall  answer_relevancy
mean      0.746941           0.895833        0.933

In [164]:
import pandas as pd
df = pd.read_json("ragas_multi_run_checkpoint.json")
for run in df['runs']:
    print(run)

{'run': 1, 'timestamp': '2026-07-11T13:32:37.164358', 'faithfulness': 0.7169841269841271, 'context_precision': 0.833333333283333, 'context_recall': 0.9666666666666661, 'answer_relevancy': 0.753388848452043}
{'run': 2, 'timestamp': '2026-07-11T13:36:29.575338', 'faithfulness': 0.7526984126984121, 'context_precision': 0.8999999999499999, 'context_recall': 0.9666666666666661, 'answer_relevancy': 0.7510937845292881}
{'run': 3, 'timestamp': '2026-07-11T13:46:13.738913', 'faithfulness': 0.7978835978835971, 'context_precision': 0.9999999999333331, 'context_recall': 0.833333333333333, 'answer_relevancy': 0.914721830883886}
{'run': 4, 'timestamp': '2026-07-11T13:58:16.526812', 'faithfulness': None, 'context_precision': None, 'context_recall': None, 'answer_relevancy': None}
{'run': 5, 'timestamp': '2026-07-11T14:02:29.803686', 'faithfulness': 0.720198412698412, 'context_precision': 0.8499999999508331, 'context_recall': 0.9666666666666661, 'answer_relevancy': 0.753499598340159}


In [ ]:
nan_count = df.isna().sum()
print(f"NaN counts per metric: {nan_count.to_dict()}")

clean_stats = df.dropna().agg(['mean', 'std'])
print(f"Number of valid runs used: {len(df.dropna())}")

ValueError: DataFrame constructor not properly called!

In [158]:
{'faithfulness': 0.8751, 'context_precision': 0.7722, 'context_recall': 0.7667, 'answer_relevancy': 0.7597}

{'faithfulness': 0.7373, 'context_precision': 0.6944, 'context_recall': 0.7778, 'answer_relevancy': 0.6259}


results_df = results.to_pandas()

worst = results_df.sort_values("context_recall").head(10)


In [159]:
worst

,user_input,retrieved_contexts,response,reference,faithfulness,context_precision,context_recall,answer_relevancy
0,This patient's bacteremia source isn't obvious...,[UNMC / NEBRASKA MEDICINE | 3 \n \nDuration ...,To determine whether a bacteremia case counts ...,Uncomplicated GN-BSI requires absence of endoc...,1.000000,0.805556,0.500000,0.882089
1,Does immunosuppression alone make a bacteremia...,[One study also reported shorter time to retur...,The provided guidelines do not explicitly stat...,"No. Per UNMC guidance, immunocompromised statu...",0.812500,1.000000,0.666667,0.961602
2,What is the recommended duration of therapy fo...,"[Herrera F, Torres D, Carena A, et al. Short C...",The recommended duration of therapy for uncomp...,7 days of therapy (oral or IV) is recommended ...,0.750000,1.000000,1.000000,1.000000
3,When should I get an ID consult for a bacterem...,[UNMC / NEBRASKA MEDICINE | 6 \n \nWhen clin...,The provided reference documents do not contai...,ID consult is recommended when clinical uncert...,0.000000,0.805556,1.000000,0.000000
4,What organisms might require a longer duration...,[UNMC / NEBRASKA MEDICINE | 3 \n \nDuration ...,Patients with Gram-negative bacteremia caused ...,"Organisms such as Salmonella, Acinetobacter, a...",0.600000,0.500000,1.000000,0.725324
5,What does 'source control' mean in the context...,[UNMC / NEBRASKA MEDICINE | 6 \n \nWhen clin...,The provided reference documents do not explic...,Source control means adequately treating or re...,1.000000,0.805556,1.000000,0.000000
6,Can a patient be switched to oral antibiotics ...,"[Additionally, consideration should be given a...","Yes, a patient can be switched to oral antibio...","Yes, patients can be transitioned to oral anti...",1.000000,1.000000,1.000000,0.950225
7,What does the BALANCE trial say about antibiot...,[2019;69(7):1091-1098. doi:10.1093/cid/ciy1054...,The BALANCE trial evaluated antibiotic treatme...,The BALANCE trial compared 7 vs 14 days of ant...,1.000000,1.000000,1.000000,0.981533
8,Is a positive blood culture always a true infe...,[It is important to \nevaluate the clinical pi...,"No, a positive blood culture is not always a t...","No, positive blood cultures can represent cont...",1.000000,0.750000,1.000000,0.990053
9,What should you check before shortening antibi...,"[Herrera F, Torres D, Carena A, et al. Short C...",Before shortening antibiotic duration for pyel...,Pyelonephritis-associated bacteremia may requi...,0.714286,0.916667,1.000000,0.967398
